# Decision model
A decision model can only choose between options, it can't extract text, so we
flatten every application/action pair into a single choice (e.g. `maps.navigate`).

In [ ]:
import json
import time
from typing import cast

from dotenv import load_dotenv
from IPython.display import JSON, display
from typesafe_sdk import Choice, ChoiceAnswer, TypeSafeClient

from demo_helpers import VALID_APPLICATIONS
from demo_helpers import plot_probabilities

load_dotenv()

In [ ]:
transcript = "Hey Google, navigate to Ara."

In [ ]:
def build_criteria(valid_applications: dict[str, dict]) -> dict[str, str]:
    """Flatten every application/action pair into one option, e.g. "maps.navigate"."""
    criteria = {}
    for app_name, app in valid_applications.items():
        for action_name in app["actions"]:
            criteria[f"{app_name}.{action_name}"] = (
                f"{app['description']}: {action_name.replace('_', ' ')}"
            )
    return criteria

## Input

In [ ]:
questions = {
    "command": Choice(
        instructions="Which application and action does the user want?",
        criteria=build_criteria(VALID_APPLICATIONS),
    ),
}
questions

## Model call

In [ ]:
with TypeSafeClient(model="jev-latest") as client:
    start = time.perf_counter()
    response = client.system_one(state=transcript, questions=questions)
    print(f"Client call took {time.perf_counter() - start:.2f}s")

## Output

In [ ]:
response

In [ ]:
answer = cast(ChoiceAnswer, response.answers["command"])
print(answer.model_dump_json(indent=2))

In [ ]:
plot_probabilities(response.answers)